In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install botasaurus global_land_mask beautifulsoup4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.1/138.1 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.2/300.2 kB 16.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.2/171.2 kB 13.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'
!apt-get -y update
!apt-get install -y google-chrome-stable

OK
Get:1 http://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,216 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,827 kB]
Get:14 http:/

In [10]:
import os
import re
import csv
import time
import random
from botasaurus.browser import browser, Driver
from botasaurus.soupify import soupify
from global_land_mask import globe
from google.colab import drive

In [11]:
CITY_NAME = "HCM"
PART_TO_RUN = 1
BATCH_SIZE = 10

BASE_DIR = '/content/drive/MyDrive/SmartSite/Step 1: Extract'
CSV_DIR = os.path.join(BASE_DIR, 'Results_CSV')
HISTORY_DIR = os.path.join(BASE_DIR, 'History_TXT')

os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(HISTORY_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(CSV_DIR, f"{CITY_NAME}_GGMap_Part{PART_TO_RUN}.csv")
HISTORY_FILE = os.path.join(HISTORY_DIR, f"{CITY_NAME}_History_Part{PART_TO_RUN}.txt")

print(f"✅ Đã cấu hình xong cho {CITY_NAME} - Luồng {PART_TO_RUN}")
print(f"✅ Lưu kết quả vào {OUTPUT_FILE}")
print(f"✅ Lưu lịch sử vào {HISTORY_FILE}")

✅ Đã cấu hình xong cho HCM - Luồng 3
✅ Lưu kết quả vào /content/drive/MyDrive/SmartSite/Step 1: Extract/Results_CSV/HCM_GGMap_Part3.csv
✅ Lưu lịch sử vào /content/drive/MyDrive/SmartSite/Step 1: Extract/History_TXT/HCM_History_Part3.txt


In [12]:
# ==============================================================================
# 1. CẤU HÌNH TỪ KHÓA
# ==============================================================================
SEARCH_KEYWORDS = ["Quán cà phê", "Coffee shop", "Cafe"]
VALID_CATEGORIES = [
    "Coffee shop", "Cafe", "Espresso bar", "Tea house", "Bubble tea store",
    "Quán cà phê", "Quán trà sữa", "Dessert shop", "Bán cà phê"
]
COFFEE_KEYWORDS = [
    "cafe", "coffee", "cà phê", "trà sữa", "milk tea",
    "tea", "trà", "roastery", "cacao", "kem", "dessert"
]
BLACKLIST_KEYWORDS = [
    "nhậu", "bia", "beer", "pub", "bar", "restaurant", "nhà hàng",
    "cơm", "bún", "phở", "mì quảng", "hủ tiếu", "cháo", "homestay",
    "hotel", "spa", "nails", "tạp hóa", "siêu thị", "pharmacy", "massage", "karaoke", "bia hơi"
]


# ==============================================================================
# 2. HÀM QUẢN LÝ LƯỚI TỰ ĐỘNG - CHUYÊN BIỆT CHO HÀ NỘI
# ==============================================================================
def get_bounds_for_part(part):
    mid_lat, mid_lng = 10.7550, 106.6000
    if part == 1:   return {"min_lat": mid_lat, "max_lat": 11.1600, "min_lng": 106.3500, "max_lng": mid_lng}
    elif part == 2: return {"min_lat": mid_lat, "max_lat": 11.1600, "min_lng": mid_lng, "max_lng": 106.8500}
    elif part == 3: return {"min_lat": 10.3500, "max_lat": mid_lat, "min_lng": 106.3500, "max_lng": mid_lng}
    elif part == 4: return {"min_lat": 10.3500, "max_lat": mid_lat, "min_lng": mid_lng, "max_lng": 106.8500}
    return {"min_lat": 10.3500, "max_lat": 11.1600, "min_lng": 106.3500, "max_lng": 106.8500}


def generate_adaptive_grid():
    points = []
    bounds = get_bounds_for_part(PART_TO_RUN)
    CORE_BOUNDS = {"min_lat": 10.7300, "max_lat": 10.8300, "min_lng": 106.6200, "max_lng": 106.7300}
    STEP_CORE, STEP_SUBURB = 0.005, 0.015

    lat = bounds["min_lat"]
    while lat <= bounds["max_lat"]:
        lng = bounds["min_lng"]
        while lng <= bounds["max_lng"]:
            if globe.is_land(lat, lng):
                in_core = (CORE_BOUNDS["min_lat"] <= lat <= CORE_BOUNDS["max_lat"]) and (CORE_BOUNDS["min_lng"] <= lng <= CORE_BOUNDS["max_lng"])
                if not in_core: points.append((lat, lng))
            lng += STEP_SUBURB
        lat += STEP_SUBURB

    lat_core = max(bounds["min_lat"], CORE_BOUNDS["min_lat"])
    max_lat_core = min(bounds["max_lat"], CORE_BOUNDS["max_lat"])
    while lat_core <= max_lat_core:
        lng_core = max(bounds["min_lng"], CORE_BOUNDS["min_lng"])
        max_lng_core = min(bounds["max_lng"], CORE_BOUNDS["max_lng"])
        while lng_core <= max_lng_core:
            if globe.is_land(lat_core, lng_core): points.append((lat_core, lng_core))
            lng_core += STEP_CORE
        lat_core += STEP_CORE

    unique_points = list(set(points))
    random.shuffle(unique_points)
    return unique_points


# ==============================================================================
# 3. QUẢN LÝ FILE
# ==============================================================================
def load_existing_ids():
    ids = set()
    try:
        if os.path.exists(OUTPUT_FILE):
            with open(OUTPUT_FILE, 'r', encoding='utf-8-sig') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    if 'place_id' in row: ids.add(row['place_id'])
            print(f"🔄 Đã load {len(ids)} quán cũ từ CSV.")
    except Exception as e:
        print(f"Lỗi đọc file CSV: {e}")
    return ids


def load_history():
    points = set()
    try:
        if os.path.exists(HISTORY_FILE):
            with open(HISTORY_FILE, 'r', encoding='utf-8') as f:
                for line in f: points.add(line.strip())
            print(f"🔄 Đã load {len(points)} điểm đã quét.")
    except Exception as e:
        print(f"Lỗi đọc file History: {e}")
    return points


def append_history(lat, lng):
    try:
        with open(HISTORY_FILE, 'a', encoding='utf-8') as f:
            f.write(f"{lat}_{lng}\n")
    except Exception as e:
        print(f"⚠️ Lỗi ghi file txt: {e}")


def save_batch(batch_data):
    if not batch_data: return
    try:
        file_exists = os.path.isfile(OUTPUT_FILE)
        with open(OUTPUT_FILE, mode='a', newline='', encoding='utf-8-sig') as f:
            fieldnames = [
                "name", "address", "phone", "category", "rating",
                "reviews_count", "status", "opening_hours", "sample_reviews",
                "lat", "lng", "grid_lat", "grid_lng", "google_url", "place_id"
            ]
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            if not file_exists:
                writer.writeheader()
            writer.writerows(batch_data)
        print(f"    💾 [SAVED] Đã lưu {len(batch_data)} quán vào Drive.")
    except Exception as e:
        print(f"⚠️ Lỗi ghi file csv: {e}")

# ==============================================================================
# 4. BỘ LỌC VÀ LẤY DỮ LIỆU
# ==============================================================================
def is_valid_cafe(name, category):
    name_lower, cat_lower = name.lower(), category.lower()
    for bad in BLACKLIST_KEYWORDS:
        if bad in name_lower or bad in cat_lower: return False
    for cat in VALID_CATEGORIES:
        if cat.lower() in cat_lower: return True
    if any(g in cat_lower for g in ["store", "cửa hàng", "shop"]):
        if any(kw in name_lower for kw in COFFEE_KEYWORDS): return True
    return False


def get_place_id(url):
    try:
        return url.split("!1s")[1].split("!")[0] if "!1s" in url else url
    except:
        return url


def extract_real_lat_lng(url):
    try:
        match = re.search(r'!3d(-?\d+\.\d+)!4d(-?\d+\.\d+)', url)
        if match:
            return float(match.group(1)), float(match.group(2))
        match = re.search(r'/@(-?\d+\.\d+),(-?\d+\.\d+)', url)
        if match:
            return float(match.group(1)), float(match.group(2))
    except:
        pass
    return None, None


def extract_hours(driver, soup):
    try:
        hours_btn = soup.select_one('[data-item-id="oh"] div[role="button"]')
        if hours_btn and hours_btn.get('aria-label'): return hours_btn.get('aria-label').replace(
            "Hide open hours for the week", "").strip()
        status_div = soup.select_one('[data-item-id="oh"]')
        if status_div: return status_div.get_text(strip=True)
    except:
        pass
    return "N/A"

# 🔥 BẢN NÂNG CẤP LẤY REVIEW THÔNG MINH
def extract_random_reviews(driver):
    try:
        review_opened = False

        # Phương án A: Tìm trực tiếp Tab "Đánh giá"
        review_btns = driver.select_all('button[role="tab"]')
        for btn in review_btns:
            label = (btn.get_attribute('aria-label') or btn.text or "").lower()
            if "review" in label or "đánh giá" in label:
                btn.click()
                review_opened = True
                break

        # Phương án B: Click thẳng vào dòng Rating
        if not review_opened:
            try:
                rating_area = driver.select('div.F7nice button') or driver.select('button[jsaction*="moreReviews"]')
                if rating_area:
                    rating_area.click()
                    review_opened = True
            except:
                pass

        if not review_opened:
            return ""

        time.sleep(random.uniform(3.0, 4.0))

        # TÌM VÙNG CUỘN TRANG
        scrollable_div = None
        scroll_selectors = [
            'div.m6QErb.DxyBCb.kA9KIf.dS8AEf',
            'div.m6QErb[aria-label]',
            'div.m6QErb',
            'div[role="main"]'
        ]
        for sel in scroll_selectors:
            try:
                el = driver.select(sel)
                if el:
                    scrollable_div = el
                    break
            except:
                continue

        # SMART SCROLL & EXPAND
        seen_texts = set()
        stuck_counter = 0
        previous_count = 0

        for _ in range(10):
            try:
                if scrollable_div:
                    driver.run_js("arguments[0].scrollTop += 800;", scrollable_div)
                else:
                    driver.run_js("window.scrollBy(0, 800);")
            except:
                try: driver.press_key('PageDown')
                except: pass

            time.sleep(random.uniform(1.5, 2.0))

            try:
                more_btns = driver.select_all('button.w8nwRe')
                for btn in more_btns:
                    try:
                        driver.run_js("arguments[0].click();", btn)
                        time.sleep(0.1)
                    except: pass
            except: pass

            soup_temp = soupify(driver)
            current_count = len(soup_temp.select('div.jftiEf, div[data-review-id]'))

            if current_count >= 15: break

            if current_count == previous_count:
                stuck_counter += 1
                if stuck_counter >= 3: break
            else:
                stuck_counter = 0

            previous_count = current_count

        time.sleep(1.0)

        # TRÍCH XUẤT DỮ LIỆU CHÍNH THỨC
        soup = soupify(driver)
        review_blocks = soup.select('div.jftiEf, div[data-review-id]')

        if not review_blocks:
            review_blocks = soup.select('div[jsinstance]')

        TEXT_SELECTORS = ['.wiI7pd', '.MyEned span', 'span[data-expandable-section]']
        temp_reviews = []

        for block in review_blocks:
            try:
                text = ""
                for sel in TEXT_SELECTORS:
                    el = block.select_one(sel)
                    if el:
                        text = el.get_text(strip=True)
                        if len(text) > 15: break

                if len(text) <= 15 or text in seen_texts: continue
                seen_texts.add(text)

                stars = "?"
                rating_el = block.select_one('span[aria-label*="star"], span[aria-label*="sao"], span[role="img"][aria-label]')
                if rating_el:
                    match = re.search(r'(\d+)', rating_el.get('aria-label', ''))
                    if match: stars = match.group(1)

                temp_reviews.append(f"[{stars} sao] {text}")
            except:
                continue

        return " || ".join(temp_reviews[:15]) if temp_reviews else ""

    except Exception as e:
        return ""


# 🔥 BẢN NÂNG CẤP GẮN TIẾNG VIỆT VÀO URL
def parse_place_full(driver, url, lat_scan, lng_scan):
    try:
        # Ép trình duyệt trả về giao diện tiếng Việt
        clean_url = url
        if "?hl=" not in clean_url and "&hl=" not in clean_url:
            clean_url += "&hl=vi" if "?" in clean_url else "?hl=vi"

        driver.get(clean_url)
        time.sleep(random.uniform(2.0, 3.0))
        soup = soupify(driver)

        try: name = soup.select_one('h1').get_text(strip=True)
        except: return None

        try: category = soup.select_one('button[jsaction*="category"]').get_text(strip=True)
        except: category = "Store"

        if not is_valid_cafe(name, category): return None

        try: address = soup.select_one('[data-item-id="address"]').get_text(strip=True)
        except: address = ""

        try: phone = soup.select_one('[data-item-id*="phone"]').get_text(strip=True)
        except: phone = ""

        try: rating = soup.select_one('div.F7nice span[aria-hidden="true"]').get_text(strip=True)
        except: rating = ""

        # Lọc số lượng đánh giá toàn diện
        try:
            reviews_count = "0"
            rating_block = soup.select_one('div.F7nice')
            if rating_block:
                full_text = rating_block.get_text(separator=' ', strip=True).lower()
                match = re.search(r'\(([\d\.,]+)\)|([\d\.,]+)\s*(?:review|đánh giá)', full_text)
                if match:
                    raw_num = match.group(1) if match.group(1) else match.group(2)
                    if raw_num: reviews_count = re.sub(r'[^\d]', '', raw_num)

            if not reviews_count or reviews_count == "0":
                for btn in soup.select('button'):
                    btn_text = btn.get_text(strip=True).lower()
                    if 'review' in btn_text or 'đánh giá' in btn_text:
                        nums = re.findall(r'[\d\.,]+', btn_text)
                        if nums:
                            reviews_count = re.sub(r'[^\d]', '', nums[0])
                            if reviews_count: break

            if not reviews_count or reviews_count == "0":
                body_text = soup.get_text(separator=' ').lower()
                match_body = re.search(r'([\d\.,]+)\s*(?:reviews|đánh giá|bài đánh giá)', body_text)
                if match_body: reviews_count = re.sub(r'[^\d]', '', match_body.group(1))

            if not reviews_count: reviews_count = "0"
        except:
            reviews_count = "0"

        status = "Closed" if "Permanently closed" in soup.get_text() or "Đã đóng cửa vĩnh viễn" in soup.get_text() else "Open"
        hours = extract_hours(driver, soup)

        # Bắt Review ở đây
        sample_reviews = extract_random_reviews(driver) if status == "Open" else ""

        real_lat, real_lng = extract_real_lat_lng(url)

        return {
            "name": name,
            "address": address,
            "phone": phone,
            "category": category,
            "rating": rating,
            "reviews_count": reviews_count,
            "status": status,
            "opening_hours": hours,
            "sample_reviews": sample_reviews,
            "lat": real_lat if real_lat else lat_scan,
            "lng": real_lng if real_lng else lng_scan,
            "grid_lat": lat_scan,
            "grid_lng": lng_scan,
            "google_url": url,
            "place_id": get_place_id(url)
        }
    except:
        return None

In [7]:
import json

@browser(block_images=True, reuse_driver=True, headless=True)
def run_quick_test(driver: Driver, data):
    print("🛠️ BẮT ĐẦU TEST CRAWL (Chỉ lấy 2 quán để kiểm tra)...")

    # Lấy thử tọa độ đầu tiên của luồng này
    test_grid = generate_adaptive_grid()
    if not test_grid:
        print("Lưới rỗng!")
        return

    lat, lng = test_grid[0]
    print(f"\n📍 Đang test tại Grid: {lat:.4f}, {lng:.4f}")

    # Chỉ dùng 1 từ khóa đầu tiên để test cho lẹ
    kw = SEARCH_KEYWORDS[0]
    driver.get(f"https://www.google.com/maps/search/{kw}/@{lat},{lng},15z")
    time.sleep(3)

    try:
        driver.scroll('div[role="feed"]')
        time.sleep(1)
    except: pass

    elements = driver.select_all('a[href*="/maps/place/"]')

    test_results = []
    for el in elements:
        url = el.get_attribute('href')
        if not url: continue

        print(f" ⏳ Đang bóc tách 1 quán...")
        place_data = parse_place_full(driver, url, lat, lng)

        if place_data:
            test_results.append(place_data)
            print(f" ✅ THÀNH CÔNG: {place_data['name']}")

        if len(test_results) >= 2: # Lấy đúng 2 cái rồi dừng
            break

    print("\n" + "="*50)
    print("📊 KẾT QUẢ DỮ LIỆU CỦA 2 QUÁN (CHECK ĐỦ 15 CỘT CHƯA NHA):")
    print("="*50)
    for res in test_results:
        print(json.dumps(res, indent=4, ensure_ascii=False))

# Chạy Test
run_quick_test()

Running
🛠️ BẮT ĐẦU TEST CRAWL (Chỉ lấy 2 quán để kiểm tra)...

📍 Đang test tại Grid: 10.6050, 106.5900
 ⏳ Đang bóc tách 1 quán...
 ✅ THÀNH CÔNG: Xanh Cafe
 ⏳ Đang bóc tách 1 quán...
 ✅ THÀNH CÔNG: Quán Xuân

📊 KẾT QUẢ DỮ LIỆU CỦA 2 QUÁN (CHECK ĐỦ 15 CỘT CHƯA NHA):
{
    "name": "Xanh Cafe",
    "address": "JH9Q+3FP, ĐT826, Long Trạch, Cần Đước, Long An, Vietnam",
    "phone": "+84 888 863 839",
    "category": "Coffee shop",
    "rating": "5.0",
    "reviews_count": "20",
    "status": "Open",
    "opening_hours": "N/A",
    "sample_reviews": "",
    "lat": 10.6180541,
    "lng": 106.5887811,
    "grid_lat": 10.60500000000001,
    "grid_lng": 106.59,
    "google_url": "https://www.google.com/maps/place/Xanh+Cafe/data=!4m7!3m6!1s0x31753500580a0829:0xa66652ad51711417!8m2!3d10.6180541!4d106.5887811!16s%2Fg%2F11yljv56hm!19sChIJKQgKWAA1dTERFxRxUa1SZqY?authuser=0&hl=en&rclk=1",
    "place_id": "0x31753500580a0829:0xa66652ad51711417"
}
{
    "name": "Quán Xuân",
    "address": "270 ấp 2, L

In [ ]:
@browser(
    headless=True,              # BẮT BUỘC True trên Colab vì không có màn hình
    window_size=(1920, 1080),   # BẮT BUỘC ép size to để Google không giấu Tab Review
    block_images=False,         # BẮT BUỘC False để Javascript của GG Maps chạy bình thường
    reuse_driver=True,
    close_on_crash=True,
    output=None
)
def run_final_crawler(driver: Driver, data):
    print(f"🚀 BẮT ĐẦU CRAWL LUỒNG {PART_TO_RUN} - HỒ CHÍ MINH...")

    crawled_ids = load_existing_ids()
    scanned_history = load_history()
    all_grid_points = generate_adaptive_grid()

    grid_points = [(lat, lng) for lat, lng in all_grid_points if f"{lat}_{lng}" not in scanned_history]
    print(f"📊 Tổng Grid luồng {PART_TO_RUN}: {len(all_grid_points)}. Cần chạy: {len(grid_points)}")

    current_batch = []

    for i, (lat, lng) in enumerate(grid_points):
        print(f"\n📍 Grid [{i + 1}/{len(grid_points)}]: {lat:.4f}, {lng:.4f}")
        found_count = 0

        for kw in SEARCH_KEYWORDS:
            try:
                driver.get(f"https://www.google.com/maps/search/{kw}/@{lat},{lng},14z")
                # Tối ưu thời gian chờ kết quả tìm kiếm
                time.sleep(random.uniform(2.5, 3.5))

                try:
                    if driver.select('div[role="feed"]'):
                        for _ in range(3):
                            driver.scroll('div[role="feed"]')
                            time.sleep(random.uniform(0.8, 1.2))
                except:
                    pass

                elements = driver.select_all('a[href*="/maps/place/"]')

                for el in elements:
                    url = el.get_attribute('href')
                    if not url: continue
                    p_id = get_place_id(url)

                    if p_id in crawled_ids or any(item['place_id'] == p_id for item in current_batch): continue

                    data = parse_place_full(driver, url, lat, lng)
                    if data:
                        current_batch.append(data)
                        crawled_ids.add(p_id)
                        found_count += 1
                        print(f" ✅ LẤY: {data['name']} | ⭐ {data['rating']} | 📝 {data['reviews_count']} đánh giá")
                    else:
                        crawled_ids.add(p_id)

                    if len(current_batch) >= BATCH_SIZE:
                        save_batch(current_batch)
                        current_batch.clear()

            except Exception as e:
                pass

        append_history(lat, lng)
        if found_count == 0: print("   ⚠️ Không có quán mới.")

    if current_batch: save_batch(current_batch)
    print("\n🎉 HOÀN TẤT VÙNG NÀY!")


if __name__ == "__main__":
    run_final_crawler()

🚀 BẮT ĐẦU CRAWL LUỒNG 1 - HỒ CHÍ MINH...
📊 Tổng Grid luồng 1: 459. Cần chạy: 459

📍 Grid [1/459]: 11.1150, 106.5300
 ✅ LẤY: Ngạn Cafe | ⭐ 4.9 | 📝 80 đánh giá
 ✅ LẤY: Tiệm cafe Tháng Năm | ⭐ 4.6 | 📝 41 đánh giá
 ✅ LẤY: Coffee Nhạc Xưa | Coffee Sân Vườn Bến Cát | ⭐ 4.7 | 📝 9 đánh giá
 ✅ LẤY: HIÊN Coffee & Tea | ⭐ 5.0 | 📝 12 đánh giá
 ✅ LẤY: Tiệm Cà Phê 1993 | ⭐ 4.5 | 📝 31 đánh giá
 ✅ LẤY: 1418 COFFEE AND MILK TEA | ⭐ 4.7 | 📝 13 đánh giá
 ✅ LẤY: Hồng Nhung Coffee - Coffee Nhân Vip | ⭐ 5.0 | 📝 1 đánh giá
 ✅ LẤY: Gold 92 Coffee | ⭐ 4.3 | 📝 27 đánh giá
 ✅ LẤY: Hà Vân Coffee | ⭐ 4.5 | 📝 11 đánh giá
 ✅ LẤY: Napoly Café & Milk Tea An Nhơn Tây, Huyện Củ Chi, TP. Hồ Chí Minh | ⭐ 4.6 | 📝 28 đánh giá
    💾 [SAVED] Đã lưu 10 quán vào Drive.
 ✅ LẤY: Café - Chòi Võng K+ | ⭐ 4.6 | 📝 20 đánh giá
 ✅ LẤY: COFFEE NGUYỄN | ⭐ 4.5 | 📝 2 đánh giá
 ✅ LẤY: Cafe Muối Chú Rắc An Tây Bến Cát | ⭐ 5.0 | 📝 1 đánh giá
 ✅ LẤY: Quán Nước mía chú Thức | ⭐ 5.0 | 📝 2 đánh giá
